# Food Lens — 3-batch: MobileNetV2 training (dataset v1)

**Run order:** first with `SMOKE_TEST = True` (synthetic images, no real data, a few minutes on CPU or GPU),
then with `SMOKE_TEST = False` on a GPU runtime.

Rules this notebook enforces:

- Inputs come from a **bundle zip** (code + `manifest_v1.csv`, no images) whose SHA-256 values are checked.
- Food-101 images are read **read-only** from `MyDrive/food101_dataset` and must match the manifest SHA-256.
- Commons kebab originals are downloaded once from `upload.wikimedia.org`, checked against the manifest
  **SHA-1** of the original file, and stored resized in `MyDrive/food-lens/commons_cache` for later runs.
- Train augmentation only; val/test preprocessing is deterministic. Class-weighted loss. Seeded.
- The best checkpoint is chosen by **val macro-F1**. Temperature and the "not sure" threshold are fitted
  **on val only**. The configuration is then **locked**, and only after that the **test split is used once**.
- Every run gets a new folder `MyDrive/food-lens/runs/<UTC time>_<commit>_seed<seed>`; nothing is overwritten.

Licensing: Food-101 allows scientific fair use only. Whether a model trained on it may back a public
deployment is **not yet clarified** — keep checkpoints private until that is decided.

In [ ]:
# 1) Settings — run the whole notebook once with SMOKE_TEST = True, then set it to False.
SMOKE_TEST = True

SEED = 42
IMG_SIZE = 224                 # model input
RESIZE = 256                   # val/test: shorter side -> 256, then center crop 224
BATCH_SIZE = 64
HEAD_EPOCHS = 3                # phase 1: backbone frozen, classifier only
FT_EPOCHS = 15                 # phase 2: whole network, cosine LR
LR_HEAD, LR_FT, WEIGHT_DECAY = 1e-3, 3e-4, 1e-4
PATIENCE = 4                   # phase 2 early stop on val macro-F1
TARGET_SELECTIVE_ACCURACY = 0.95   # val accuracy required on predictions that are not "not sure"
N_BOOT = 1000                  # bootstrap resamples for test confidence intervals

DRIVE_ROOT = "/content/drive/MyDrive"
DATASET_DIR = f"{DRIVE_ROOT}/food101_dataset"                 # read-only, never written
PROJECT_DIR = f"{DRIVE_ROOT}/food-lens"
BUNDLE_ZIP = f"{PROJECT_DIR}/bundles/food_lens_bundle_XXXXXXX.zip"  # <- set to the uploaded bundle
COMMONS_CACHE = f"{PROJECT_DIR}/commons_cache"               # verified Commons images (reused)
RUNS_ROOT = f"{PROJECT_DIR}/runs"                             # one new folder per run
LOCAL_CACHE = "/content/food_lens_cache"                      # local disk, per session
USER_AGENT = "FoodLensTraining/0.1 (+https://github.com/shohruhinomjonov691-hub)"

In [ ]:
# 2) Environment, versions, seeding.
import csv, hashlib, io, json, os, platform, random, shutil, sys, tempfile, time, urllib.request, zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # needed for deterministic cuBLAS
import numpy as np
import PIL
import torch
import torchvision
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

IN_COLAB = "google.colab" in sys.modules
SMOKE_TEST = bool(int(os.environ.get("FOOD_LENS_SMOKE", "1" if SMOKE_TEST else "0")))
BUNDLE_ZIP = os.environ.get("FOOD_LENS_BUNDLE", BUNDLE_ZIP)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 2 if IN_COLAB else 0


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


seed_everything(SEED)
ENV = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
       "numpy": np.__version__, "pillow": PIL.__version__, "device": DEVICE,
       "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
       "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(), "in_colab": IN_COLAB,
       "platform": platform.platform()}
print(json.dumps(ENV, indent=1))
print("SMOKE_TEST =", SMOKE_TEST)
if not SMOKE_TEST and DEVICE != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU before the real run.")

In [ ]:
# 3) Google Drive (Colab only). The dataset folder is only ever opened with "rb".
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
if SMOKE_TEST:
    # Smoke runs never touch the real dataset, cache or runs folder.
    SMOKE_ROOT = tempfile.mkdtemp(prefix="food_lens_smoke_")
    DATASET_DIR = f"{SMOKE_ROOT}/food101_dataset"
    COMMONS_CACHE = f"{SMOKE_ROOT}/commons_cache"
    RUNS_ROOT = f"{SMOKE_ROOT}/runs"
    LOCAL_CACHE = f"{SMOKE_ROOT}/local_cache"
    HEAD_EPOCHS, FT_EPOCHS, BATCH_SIZE, N_BOOT = 1, 2, 16, 200
    print("Smoke folders under", SMOKE_ROOT)

In [ ]:
# 4) Bundle: unpack, verify every file against BUNDLE.json, import the repo code.
assert os.path.exists(BUNDLE_ZIP), f"Bundle not found: {BUNDLE_ZIP} (set BUNDLE_ZIP in cell 1)"
BUNDLE_SHA256 = hashlib.sha256(open(BUNDLE_ZIP, "rb").read()).hexdigest()
CODE_DIR = tempfile.mkdtemp(prefix="food_lens_code_")
with zipfile.ZipFile(BUNDLE_ZIP) as z:
    z.extractall(CODE_DIR)
BUNDLE = json.load(open(f"{CODE_DIR}/BUNDLE.json"))
for rel, expected in BUNDLE["files"].items():
    actual = hashlib.sha256(open(f"{CODE_DIR}/{rel}", "rb").read()).hexdigest()
    assert actual == expected, f"bundle file changed: {rel}"
sys.path.insert(0, CODE_DIR)
from food_lens import train_utils as tu
from food_lens.classes import CLASSES, CLASS_TO_INDEX, VARIANTS
from food_lens.imaging import load_rgb

COMMIT = BUNDLE["commit"]
print("Bundle OK:", os.path.basename(BUNDLE_ZIP), "commit", COMMIT[:7], "| dirty" if BUNDLE.get("dirty") else "")
print("Classes (model output order):", CLASSES)

In [ ]:
# 5) Manifest rows: the real manifest, or a small synthetic dataset for the smoke test.
def synthetic_dataset(root):
    # Fake Food-101 folders + fake Commons originals (file:// URLs) with real checksums.
    rng = np.random.default_rng(SEED)
    colors = {"cheesecake": (240, 230, 200), "chocolate_cake": (90, 50, 30), "hamburger": (200, 140, 60),
              "hot_dog": (220, 90, 60), "pizza": (230, 60, 40), "skewered": (150, 80, 40),
              "minced_skewer": (120, 60, 50), "doner_shawarma": (210, 170, 120)}
    rows, plan = [], ["train"] * 8 + ["val"] * 2 + ["test"] * 2
    os.makedirs(f"{root}/commons_originals", exist_ok=True)
    for variant, rgb in colors.items():
        target = "kebab" if variant in VARIANTS["kebab"] else ("dessert" if variant in VARIANTS["dessert"] else variant)
        for i, split in enumerate(plan):
            arr = np.clip(np.array(rgb) + rng.normal(0, 25, (240, 320, 3)), 0, 255).astype(np.uint8)
            arr[60 + 10 * (i % 4):140, 80:240] = 255 - np.array(rgb)          # a class-coloured shape
            buf = io.BytesIO()
            Image.fromarray(arr).save(buf, "JPEG" if target != "kebab" else "PNG")
            data = buf.getvalue()
            common = {"target": target, "variant": variant, "split": split, "group_id": f"s{variant}{i}"}
            if target == "kebab":
                path = f"{root}/commons_originals/{variant}_{i}.png"
                open(path, "wb").write(data)
                rows.append({**common, "sample_id": f"commons:{9000000 + 100 * list(colors).index(variant) + i}",
                             "source": "commons", "location": "file://" + path,
                             "checksum": "sha1:" + hashlib.sha1(data).hexdigest()})
            else:
                os.makedirs(f"{DATASET_DIR}/{variant}", exist_ok=True)
                open(f"{DATASET_DIR}/{variant}/{i}.jpg", "wb").write(data)
                rows.append({**common, "sample_id": f"food101:{variant}/{i}.jpg", "source": "food101",
                             "location": f"{variant}/{i}.jpg", "checksum": "sha256:" + hashlib.sha256(data).hexdigest()})
    return rows


if SMOKE_TEST:
    ROWS = synthetic_dataset(SMOKE_ROOT)
else:
    with open(f"{CODE_DIR}/manifests/manifest_v1.csv", newline="", encoding="utf-8") as f:
        ROWS = list(csv.DictReader(f))
assert all(r["target"] in CLASS_TO_INDEX for r in ROWS)
COUNTS = {s: {c: sum(1 for r in ROWS if r["split"] == s and r["target"] == c) for c in CLASSES}
          for s in ("train", "val", "test")}
VARIANT_COUNTS = Counter((r["split"], r["target"], r["variant"]) for r in ROWS)
print(len(ROWS), "images")
for s, c in COUNTS.items():
    print(f"  {s:5s}", c)

In [ ]:
# 6) Verify every image against the manifest and build the local cache (shorter side = RESIZE).
os.makedirs(LOCAL_CACHE, exist_ok=True)
os.makedirs(COMMONS_CACHE, exist_ok=True)
INDEX_PATH = f"{COMMONS_CACHE}/index.json"
COMMONS_INDEX = json.load(open(INDEX_PATH)) if os.path.exists(INDEX_PATH) else {}


def resize_short(img, side):
    w, h = img.size
    s = side / min(w, h)
    return img.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BICUBIC) if s < 1 else img


def local_path(row):
    return f"{LOCAL_CACHE}/{hashlib.sha1(row['sample_id'].encode()).hexdigest()}.jpg"


def save_local(row, img):
    resize_short(img, RESIZE).save(local_path(row), "JPEG", quality=95)


def prepare_food101(row):
    with open(os.path.join(DATASET_DIR, row["location"]), "rb") as f:   # read-only
        data = f.read()
    if not tu.checksum_ok(data, row["checksum"]):
        raise ValueError(f"checksum mismatch: {row['location']}")
    save_local(row, load_rgb(data))


def fetch_original(url):
    for attempt in range(4):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except Exception as exc:   # network hiccup: back off and retry
            last = exc
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"download failed: {url}: {last}")


def prepare_commons(row):
    # Cached file is used only if it was made from an original whose SHA-1 matched this manifest row.
    sid, cache_file = row["sample_id"], f"{COMMONS_CACHE}/{row['sample_id'].replace(':', '_')}.jpg"
    entry = COMMONS_INDEX.get(sid)
    if entry and entry["original_checksum"] == row["checksum"] and os.path.exists(cache_file) \
            and tu.file_digest(cache_file, "sha256") == entry["cached_sha256"]:
        img = Image.open(cache_file).convert("RGB")
    else:
        data = fetch_original(row["location"])
        if not tu.checksum_ok(data, row["checksum"]):
            raise ValueError(f"Commons original does not match the manifest SHA-1: {sid} {row['location']}")
        img = resize_short(load_rgb(data), 2 * RESIZE)
        img.save(cache_file, "JPEG", quality=95)
        COMMONS_INDEX[sid] = {"original_checksum": row["checksum"], "url": row["location"],
                              "cached_sha256": tu.file_digest(cache_file, "sha256"),
                              "verified_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
        tu.write_json(INDEX_PATH, COMMONS_INDEX, overwrite=True)
        time.sleep(0 if SMOKE_TEST else 0.5)
    save_local(row, img)


t0, failures = time.time(), []
food = [r for r in ROWS if r["source"] == "food101"]
commons = [r for r in ROWS if r["source"] == "commons"]


def safe(fn, row):
    try:
        fn(row)
    except Exception as exc:
        failures.append(f"{row['sample_id']}: {type(exc).__name__}: {exc}")


with ThreadPoolExecutor(max_workers=16) as pool:
    list(pool.map(lambda r: safe(prepare_food101, r), food))
print(f"Food-101: {len(food)} files verified in {time.time() - t0:.0f}s")
for i, r in enumerate(commons, 1):
    safe(prepare_commons, r)
    if i % 50 == 0:
        print(f"  Commons {i}/{len(commons)}")
print(f"Commons: {len(commons)} files ready in {time.time() - t0:.0f}s total")
assert not failures, "Stop: these images could not be verified:\n" + "\n".join(failures[:30])

In [ ]:
# 7) Data loaders. Train: augmentation. Val/test: deterministic resize + center crop.
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
TRAIN_TF = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(3 / 4, 4 / 3)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
EVAL_TF = transforms.Compose([
    transforms.Resize(RESIZE), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
PREPROCESSING = {
    "decode": "food_lens.imaging.load_rgb: full decode, EXIF orientation, transparency onto white, RGB",
    "cache": f"shorter side resized to {RESIZE} (bicubic), JPEG q95",
    "eval": f"Resize({RESIZE}) -> CenterCrop({IMG_SIZE}) -> ToTensor -> Normalize(mean={MEAN}, std={STD})",
    "train": f"RandomResizedCrop({IMG_SIZE}, scale=(0.6,1), ratio=(3/4,4/3)) -> HorizontalFlip(0.5) -> "
             "ColorJitter(0.2,0.2,0.2,0.02) -> ToTensor -> Normalize(same)",
    "inference_note": "The API must use the eval pipeline exactly (after the user's crop).",
}


class ManifestDataset(Dataset):
    def __init__(self, rows, tf):
        self.rows, self.tf = rows, tf

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        with Image.open(local_path(r)) as im:
            return self.tf(im.convert("RGB")), CLASS_TO_INDEX[r["target"]], i


def worker_init(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


SPLIT_ROWS = {s: sorted([r for r in ROWS if r["split"] == s], key=lambda r: r["sample_id"]) for s in ("train", "val", "test")}
gen = torch.Generator().manual_seed(SEED)
LOADERS = {
    "train": DataLoader(ManifestDataset(SPLIT_ROWS["train"], TRAIN_TF), batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, worker_init_fn=worker_init, generator=gen, drop_last=False),
    "val": DataLoader(ManifestDataset(SPLIT_ROWS["val"], EVAL_TF), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS),
}

In [ ]:
# 8) Run folder (new, never overwritten) and the run configuration.
RUN_NAME = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + f"_{COMMIT[:7]}_seed{SEED}" + ("_smoke" if SMOKE_TEST else "")
RUN_DIR = tu.new_run_dir(RUNS_ROOT, RUN_NAME)
TRAIN_COUNTS = [COUNTS["train"][c] for c in CLASSES]
CLASS_WEIGHTS = tu.class_weights(TRAIN_COUNTS)
WEIGHTS_NAME = None if SMOKE_TEST else "MobileNet_V2_Weights.IMAGENET1K_V2"   # smoke: no download
CONFIG = {
    "run_name": RUN_NAME, "smoke_test": SMOKE_TEST, "seed": SEED, "bundle": {**BUNDLE, "zip_sha256": BUNDLE_SHA256},
    "model": "torchvision mobilenet_v2", "pretrained_weights": WEIGHTS_NAME, "img_size": IMG_SIZE, "resize": RESIZE,
    "batch_size": BATCH_SIZE, "head_epochs": HEAD_EPOCHS, "ft_epochs": FT_EPOCHS, "lr_head": LR_HEAD, "lr_ft": LR_FT,
    "weight_decay": WEIGHT_DECAY, "patience": PATIENCE, "optimizer": "AdamW", "schedule": "cosine (phase 2)",
    "loss": "CrossEntropyLoss(weight=class_weights)", "class_weights": dict(zip(CLASSES, CLASS_WEIGHTS.tolist())),
    "selection": "best val macro-F1 (tie: lower val loss)", "target_selective_accuracy": TARGET_SELECTIVE_ACCURACY,
    "counts": COUNTS, "variant_counts": {"/".join(k): v for k, v in sorted(VARIANT_COUNTS.items())},
    "environment": ENV, "determinism": "seeded; cudnn deterministic; GPU results can still differ slightly between runs",
    "data_license_note": "Food-101: scientific fair use only; public deployment terms not yet clarified",
}
tu.write_json(f"{RUN_DIR}/config.json", CONFIG)
tu.write_json(f"{RUN_DIR}/class_mapping.json", {"classes": list(CLASSES), "class_to_index": CLASS_TO_INDEX,
                                                 "variants": {k: list(v) for k, v in VARIANTS.items()}})
tu.write_json(f"{RUN_DIR}/preprocessing.json", PREPROCESSING)
print("Run folder:", RUN_DIR)
print("Class weights:", CONFIG["class_weights"])

In [ ]:
# 9) Model and training: phase 1 (classifier only), phase 2 (all layers). Best = val macro-F1.
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2

model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2 if WEIGHTS_NAME else None)
model.classifier[1] = nn.Linear(model.last_channel, len(CLASSES))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE))


@torch.no_grad()
def predict(loader):
    model.eval()
    logits, labels = [], []
    for x, y, _ in loader:
        logits.append(model(x.to(DEVICE)).float().cpu())
        labels.append(y)
    return torch.cat(logits).numpy(), torch.cat(labels).numpy()


def run_epoch(optimizer):
    model.train()
    total, n = 0.0, 0
    for x, y, _ in LOADERS["train"]:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total, n = total + loss.item() * len(y), n + len(y)
    return total / n


best = {"macro_f1": -1.0, "val_loss": float("inf")}
history, epoch = [], 0
BEST_PATH = f"{RUN_DIR}/best.pt"


def evaluate_and_maybe_save(phase, train_loss):
    global best
    logits, labels = predict(LOADERS["val"])
    val_loss = float(criterion(torch.tensor(logits, device=DEVICE), torch.tensor(labels, device=DEVICE)).item())
    s = tu.summary(labels, logits.argmax(1), len(CLASSES))
    row = {"epoch": epoch, "phase": phase, "train_loss": round(train_loss, 5), "val_loss": round(val_loss, 5),
           "val_accuracy": round(s["accuracy"], 5), "val_macro_f1": round(s["macro_f1"], 5)}
    improved = (s["macro_f1"], -val_loss) > (best["macro_f1"], -best["val_loss"])
    if improved:
        best = {"macro_f1": s["macro_f1"], "val_loss": val_loss, "epoch": epoch, "phase": phase}
        torch.save({"state_dict": model.state_dict(), "classes": list(CLASSES), "epoch": epoch, "phase": phase,
                    "val_macro_f1": s["macro_f1"], "arch": "mobilenet_v2", "img_size": IMG_SIZE}, BEST_PATH + ".tmp")
        os.replace(BEST_PATH + ".tmp", BEST_PATH)
    history.append({**row, "best": improved})
    print(row, "<- best" if improved else "")
    return improved


for p in model.features.parameters():
    p.requires_grad = False
opt = torch.optim.AdamW(model.classifier.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
for _ in range(HEAD_EPOCHS):
    epoch += 1
    evaluate_and_maybe_save("head", run_epoch(opt))

for p in model.parameters():
    p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=LR_FT, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
stale = 0
for _ in range(FT_EPOCHS):
    epoch += 1
    stale = 0 if evaluate_and_maybe_save("finetune", run_epoch(opt)) else stale + 1
    sched.step()
    if stale >= PATIENCE:
        print("Early stop: no val macro-F1 gain for", PATIENCE, "epochs")
        break

with open(f"{RUN_DIR}/history.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0]))
    w.writeheader()
    w.writerows(history)
print("Best:", best)

In [ ]:
# 10) Calibration on VAL only: temperature scaling + "not sure" threshold. Then lock the final config.
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
val_logits, val_labels = predict(LOADERS["val"])
T = tu.fit_temperature(val_logits, val_labels)
val_probs = tu.softmax(val_logits, T)
THRESHOLD = tu.select_threshold(val_probs, val_labels, TARGET_SELECTIVE_ACCURACY)
CALIBRATION = {
    "temperature": T, "fitted_on": "val",
    "val_nll_before": tu.nll(val_logits, val_labels), "val_nll_after": tu.nll(val_logits, val_labels, T),
    "val_ece_before": tu.ece(tu.softmax(val_logits), val_labels), "val_ece_after": tu.ece(val_probs, val_labels),
    "threshold": THRESHOLD,
    "rule": "top-1 calibrated probability < threshold -> 'not sure'",
}
VAL_METRICS = tu.summary(val_labels, val_probs.argmax(1), len(CLASSES))
tu.write_json(f"{RUN_DIR}/calibration.json", CALIBRATION)
tu.write_json(f"{RUN_DIR}/val_metrics.json", {**VAL_METRICS, "classes": list(CLASSES), "best": best})
FINAL = {"checkpoint": "best.pt", "checkpoint_sha256": tu.file_digest(BEST_PATH, "sha256"),
         "temperature": T, "threshold": THRESHOLD["threshold"], "best_epoch": best["epoch"],
         "locked_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
tu.write_json(f"{RUN_DIR}/final.json", FINAL)   # refuses to overwrite: the choice is now fixed
print(json.dumps(CALIBRATION, indent=1))
print("Locked:", FINAL)

In [ ]:
# 11) TEST — used once, only with the locked checkpoint, temperature and threshold.
assert os.path.exists(f"{RUN_DIR}/final.json"), "lock the final config (cell 10) first"
assert not os.path.exists(f"{RUN_DIR}/test_metrics.json"), "test was already evaluated for this run"
FINAL = json.load(open(f"{RUN_DIR}/final.json"))
assert tu.file_digest(BEST_PATH, "sha256") == FINAL["checkpoint_sha256"], "checkpoint changed after locking"
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE)["state_dict"])
test_rows = SPLIT_ROWS["test"]
test_logits, test_labels = predict(DataLoader(ManifestDataset(test_rows, EVAL_TF), batch_size=BATCH_SIZE,
                                              num_workers=NUM_WORKERS))
probs = tu.softmax(test_logits, FINAL["temperature"])
pred = probs.argmax(1)
K = len(CLASSES)
overall = tu.summary(test_labels, pred, K)
kebab_mask = test_labels == CLASS_TO_INDEX["kebab"]
per_variant = []
for (target, variant) in sorted({(r["target"], r["variant"]) for r in test_rows}):
    m = np.array([(r["target"], r["variant"]) == (target, variant) for r in test_rows])
    counts = Counter(CLASSES[p] for p in pred[m])
    per_variant.append({"target": target, "variant": variant, "n": int(m.sum()),
                        "recall": float((pred[m] == test_labels[m]).mean()),
                        "predicted_as": dict(sorted(counts.items())), "small_n": bool(m.sum() < 30)})
TEST_METRICS = {
    **overall, "classes": list(CLASSES),
    "per_class": {c: {"precision": overall["precision"][i], "recall": overall["recall"][i],
                      "f1": overall["f1"][i], "support": overall["support"][i]} for i, c in enumerate(CLASSES)},
    "per_variant": per_variant,
    "selective_at_threshold": {**tu.selective(probs, test_labels, FINAL["threshold"]), "threshold": FINAL["threshold"]},
    "bootstrap_95ci": tu.bootstrap_ci(test_labels, pred, K, n_boot=N_BOOT, seed=SEED, mask=kebab_mask),
    "bootstrap_subset": "kebab recall",
    "note": "test used once after locking final.json; small per-variant n means wide uncertainty",
}
tu.write_json(f"{RUN_DIR}/test_metrics.json", TEST_METRICS)
with open(f"{RUN_DIR}/confusion_matrix.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["true \\ pred"] + list(CLASSES))
    for c, row in zip(CLASSES, overall["confusion_matrix"]):
        w.writerow([c] + row)
with open(f"{RUN_DIR}/test_predictions.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["sample_id", "target", "variant", "predicted", "confidence", "not_sure"])
    for r, p_, pr in zip(test_rows, pred, probs):
        w.writerow([r["sample_id"], r["target"], r["variant"], CLASSES[p_], f"{pr.max():.4f}",
                    pr.max() < FINAL["threshold"]])

ci = TEST_METRICS["bootstrap_95ci"]
lines = [f"# Food Lens test — {RUN_NAME}",
         f"- accuracy {overall['accuracy']:.3f} (95% CI {ci['accuracy'][0]:.3f}–{ci['accuracy'][1]:.3f}), "
         f"macro-F1 {overall['macro_f1']:.3f} (95% CI {ci['macro_f1'][0]:.3f}–{ci['macro_f1'][1]:.3f}), n={overall['n']}",
         f"- temperature {FINAL['temperature']:.3f}; threshold {FINAL['threshold']:.3f} -> "
         f"not sure {TEST_METRICS['selective_at_threshold']['uncertain_rate']:.1%}, "
         f"accuracy on the rest {TEST_METRICS['selective_at_threshold']['selective_accuracy']:.3f}",
         "", "| class | precision | recall | F1 | n |", "|---|---:|---:|---:|---:|"]
lines += [f"| {c} | {v['precision']:.3f} | {v['recall']:.3f} | {v['f1']:.3f} | {v['support']} |"
          for c, v in TEST_METRICS["per_class"].items()]
lines += ["", "| class / variant | n | recall | predicted as |", "|---|---:|---:|---|"]
lines += [f"| {v['target']} / {v['variant']} | {v['n']}{' (small)' if v['small_n'] else ''} | {v['recall']:.3f} | "
          f"{v['predicted_as']} |" for v in per_variant]
lines += ["", "Confusion matrix (rows = true):", "| | " + " | ".join(CLASSES) + " |", "|---" * (K + 1) + "|"]
lines += [f"| {c} | " + " | ".join(map(str, row)) + " |" for c, row in zip(CLASSES, overall["confusion_matrix"])]
open(f"{RUN_DIR}/summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

In [ ]:
# 12) Smoke test checks (only when SMOKE_TEST = True).
if SMOKE_TEST:
    expected = ["config.json", "class_mapping.json", "preprocessing.json", "best.pt", "history.csv",
                "calibration.json", "val_metrics.json", "final.json", "test_metrics.json",
                "confusion_matrix.csv", "test_predictions.csv", "summary.md"]
    missing = [f for f in expected if not os.path.exists(f"{RUN_DIR}/{f}")]
    assert not missing, missing
    try:                                     # the test split cannot be evaluated twice
        tu.write_json(f"{RUN_DIR}/test_metrics.json", {})
        raise AssertionError("second test write was allowed")
    except FileExistsError:
        pass
    try:                                     # an existing run folder is never reused
        tu.new_run_dir(RUNS_ROOT, RUN_NAME)
        raise AssertionError("run folder was reused")
    except FileExistsError:
        pass
    bad = dict(next(r for r in ROWS if r["source"] == "commons"), sample_id="commons:tampered",
               checksum="sha1:" + "0" * 40)
    try:                                     # a Commons file with the wrong SHA-1 is refused
        prepare_commons(bad)
        raise AssertionError("tampered Commons file accepted")
    except ValueError:
        pass
    assert "commons:tampered" not in COMMONS_INDEX
    assert json.load(open(f"{RUN_DIR}/final.json"))["checkpoint_sha256"] == tu.file_digest(BEST_PATH, "sha256")
    print("SMOKE TEST PASSED — set SMOKE_TEST = False in cell 1 for the real run.")
else:
    print("Real run finished:", RUN_DIR)

## After the real run, send back

- `summary.md` from the run folder (or the printed output of cell 11),
- `config.json`, `calibration.json`, `val_metrics.json`, `test_metrics.json`, `history.csv`.

Do **not** share `best.pt` publicly (Food-101 terms for public use are not clarified yet).